In [12]:
# 读取目标图像
target_image_names = ["gray", "face", "mist", "mask"]
target_image_paths = ["/data1/humw/Codes/FaceOff/target_images/gray/204_Gray_Uniformity_1.png",
                      "/data1/humw/Datasets/VGGFace2/n000138/set_B/0035_01.png",
                      "/data1/humw/Codes/FaceOff/target_images/mist/MIST_0.png",
                      "/data1/humw/Codes/FaceOff/target_images/yingbu/yingbu0.png"
                      # "/data1/humw/Codes/My-Anti-DreamBooth/target_images/crop_yingbu.png"
                      ]

# # 获取每张图像的image caption
# # 获取方式为输入DeepSeek，指令为：provide image caption for the image, and only return the caption
# target_image_captions = [
#     "A plain light gray background with no subjects, patterns, or text – a minimalist empty space.",
#     "A woman with neatly styled brown hair and bangs wears an elegant embroidered traditional Chinese dress with intricate collar details, holding a green plush item against a clean white background.",
#     "A monochrome pattern of repeating white geometric 'T'-shaped motifs arranged in orderly rows and columns against a solid black background.",
#     "A vibrant yellow Peking Opera mask with bold black outlines, intricate red and white decorative patterns around the eyes, swirling cheek designs, and a prominent black circle on the forehead, exuding traditional artistry and dramatic intensity."
# ]
# 获取方式为输入DeepSeek，指令为：provide image caption for the image, use template "a photo of a [class name]", and only return the caption
# target_image_captions = [
#     "photo of a gray square",
#     "photo of a woman in traditional Chinese dress",
#     "photo of a geometric pattern",
#     "photo of a Chinese opera mask"
# ]
target_image_captions = [
    "portrait",
]

In [13]:
# 注释：本代码片段展示了如何使用CLIP模型进行图像和文本的匹配。

import torch
import clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# target_image_paths每个图像路径与target_image_captions每个图像描述一一对应，计算图文匹配余弦相似度
img_txt_sim = []
for idx in range(len(target_image_paths)):
    image = preprocess(Image.open(target_image_paths[idx])).unsqueeze(0).to(device)
    text = clip.tokenize(target_image_captions).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image)
        text_features = model.encode_text(text)
        print(image_features.shape, text_features.shape)
        # 计算余弦相似度
        cos_sim = torch.cosine_similarity(image_features, text_features)
        print("Cosine similarity:", cos_sim)
        # 代码注释：此处用的是CLIP模型自带的多模态匹配，而非余弦相似度
        # logits_per_image, logits_per_text = model(image, text)
        # probs = logits_per_image.softmax(dim=-1).cpu().numpy()

    # print("Label probs:", probs)  # prints: [[0.9927937  0.00421068 0.00299572]]

torch.Size([1, 512]) torch.Size([1, 512])
Cosine similarity: tensor([0.2184], device='cuda:0', dtype=torch.float16)
torch.Size([1, 512]) torch.Size([1, 512])
Cosine similarity: tensor([0.2296], device='cuda:0', dtype=torch.float16)
torch.Size([1, 512]) torch.Size([1, 512])
Cosine similarity: tensor([0.2065], device='cuda:0', dtype=torch.float16)
torch.Size([1, 512]) torch.Size([1, 512])
Cosine similarity: tensor([0.2155], device='cuda:0', dtype=torch.float16)


In [14]:
# 加载数据集
from PIL import Image
import json
from pathlib import Path
from torchvision import transforms

dataset_dir = '/data1/humw/Datasets/VGGFace2'

def load_data(data_dir, image_size=512, resample=2):
    import numpy as np
    def image_to_numpy(image):
        return np.array(image).astype(np.uint8)
    # more robust loading to avoid loaing non-image files
    images = [] 
    for i in list(Path(data_dir).iterdir()):
        if not i.suffix in [".jpg", ".png", ".jpeg"]:
            continue
        else:
            images.append(image_to_numpy(Image.open(i).convert("RGB")))
    images = [Image.fromarray(i).resize((image_size, image_size), resample) for i in images]
    images = np.stack(images)
    # from B x H x W x C to B x C x H x W
    images = torch.from_numpy(images).permute(0, 3, 1, 2).float()
    assert images.shape[-1] == images.shape[-2]
    return images

train_aug = [
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(224),
    ]
tensorize_and_normalize = [
    transforms.Normalize([0.5*255]*3,[0.5*255]*3),
]
all_trans = train_aug + tensorize_and_normalize
all_trans = transforms.Compose(all_trans)
    
# 加载模型
from transformers.models.clip.modeling_clip import CLIPVisionModelWithProjection
import torch
import os
import torch.nn.functional as F

device = "cuda:0"
torch_dtype = torch.bfloat16
pretrained_model_name_or_path = '/data1/humw/Pretrains/clip-vit-large-patch14'
model = CLIPVisionModelWithProjection.from_pretrained(pretrained_model_name_or_path).to(device, dtype=torch_dtype)
model.to(torch_dtype)


CLIPVisionModelWithProjection(
  (vision_model): CLIPVisionTransformer(
    (embeddings): CLIPVisionEmbeddings(
      (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
      (position_embedding): Embedding(257, 1024)
    )
    (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-23): 24 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=1024, out_featur

In [ ]:
gray_dir = "/data1/humw/Codes/FaceOff/target_images/gray"
gray_images = load_data(data_dir=gray_dir, image_size=512, resample=2)
tran_gray_images = gray_images.transform(gray_images)

mist_dir = "/data1/humw/Codes/FaceOff/target_images/mist"
mist_images = load_data(data_dir=mist_dir, image_size=512, resample=2)
tran_mist_images = mist_images.transform(mist_images)

yingbu_dir = "/data1/humw/Codes/FaceOff/target_images/yingbu"
yingbu_images = load_data(data_dir=yingbu_dir, image_size=512, resample=2)
tran_yingbu_images = yingbu_images.transform(yingbu_images)

gray_embeds = model(tran_gray_images, output_hidden_states=True).hidden_states[-2]
mist_embeds = model(tran_mist_images, output_hidden_states=True).hidden_states[-2]
yingbu_embeds = model(tran_yingbu_images, output_hidden_states=True).hidden_states[-2]



In [2]:
# 计算图像的RMS Contrast
import cv2
import numpy as np

def compute_rms_contrast_per_channel(image_path):
    # 使用 OpenCV 读取图像（默认为 BGR 格式）
    img = cv2.imread(image_path)

    if img is None:
        raise ValueError(f"Failed to load image: {image_path}")

    # 拆分通道（B, G, R）
    channels = cv2.split(img)

    # 存储每个通道的 RMS contrast
    rms_contrasts = {}

    for i, channel_name in enumerate(['B', 'G', 'R']):
        # 转为 float32 以避免溢出
        channel = channels[i].astype(np.float32)

        mean = np.mean(channel)
        rms = np.sqrt(np.mean((channel - mean) ** 2))
        rms_contrasts[channel_name] = rms

    return rms_contrasts

# 示例使用
for idx in range(len(target_image_paths)):
    image_path = target_image_paths[idx]
    image_name = target_image_names[idx]
    print(f'Processing image: {image_name}')
    contrast_dict = compute_rms_contrast_per_channel(image_path)

    for ch, val in contrast_dict.items():
        print(f'RMS Contrast ({ch} channel): {val:.4f}')

    avg_rms = np.mean(list(contrast_dict.values()))
    print(f'Average RMS Contrast: {avg_rms:.4f}')
    
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean = np.mean(gray)
    rms = np.sqrt(np.mean((gray - mean) ** 2))
    print(f'RMS Contrast (Gray): {rms:.4f}')


Processing image: gray
RMS Contrast (B channel): 0.0000
RMS Contrast (G channel): 0.0000
RMS Contrast (R channel): 0.0000
Average RMS Contrast: 0.0000
RMS Contrast (Gray): 0.0000
Processing image: face
RMS Contrast (B channel): 89.2092
RMS Contrast (G channel): 83.4311
RMS Contrast (R channel): 82.0771
Average RMS Contrast: 84.9058
RMS Contrast (Gray): 82.4229
Processing image: mist
RMS Contrast (B channel): 110.0322
RMS Contrast (G channel): 110.0322
RMS Contrast (R channel): 110.0322
Average RMS Contrast: 110.0322
RMS Contrast (Gray): 110.0322
Processing image: mask
RMS Contrast (B channel): 92.1284
RMS Contrast (G channel): 88.7663
RMS Contrast (R channel): 85.7962
Average RMS Contrast: 88.8970
RMS Contrast (Gray): 85.2875


RMS Contrast，脸谱图像并不如人脸图像，猜测是因为脸谱图像的背景是纯白，而人脸图像的背景是复杂的。
解决方案：裁剪脸谱图像和人脸图像，留下脸谱图像的主体部分，再进行对比度增强。


In [5]:
import cv2
import numpy as np

def michelson_contrast_gray(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    I_max = float(np.max(gray))
    I_min = float(np.min(gray))
    contrast = (I_max - I_min) / (I_max + I_min + 1e-5)
    return contrast

# 示例使用
for idx in range(len(target_image_paths)):
    image_path = target_image_paths[idx]
    image_name = target_image_names[idx]
    print(f'Processing image: {image_name}')
    mi_contrast = michelson_contrast_gray(image_path)
    print(f'Michelson contrast for {image_name}: {mi_contrast}')

Processing image: gray
Michelson contrast for gray: 0.0
Processing image: face
Michelson contrast for face: 0.9999999603174619
Processing image: mist
Michelson contrast for mist: 0.9999999607843153
Processing image: mask
Michelson contrast for mask: 0.9999999607843153


- Contrast Ratio（对比度比值） 是图像亮度最亮与最暗区域之间的比值，常用于显示设备性能测评（如 HDR 亮度表现）或图像亮度动态范围分析。

In [2]:
import cv2
import numpy as np

def contrast_ratio_gray(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    I_max = np.max(gray)
    I_min = np.min(gray)

    ratio = I_max / (I_min + 1e-5)  # 防止除以0
    return ratio

# 示例使用
for idx in range(len(target_image_paths)):
    image_path = target_image_paths[idx]
    image_name = target_image_names[idx]
    print(f'Processing image: {image_name}')
    contrast_ratio = contrast_ratio_gray(image_path)

    print(f'Contrast Ratio: {contrast_ratio}')



Processing image: gray
Contrast Ratio: 0.9999999218750061
Processing image: face
Contrast Ratio: 25199999.999999996
Processing image: mist
Contrast Ratio: 25499999.999999996
Processing image: mask
Contrast Ratio: 25499999.999999996


## 对比度计算
- 指标选择：RMS Contrast
- 注意事项：人脸图像，计算所有人脸的对比度然后取均值
- 脸谱，分别计算裁剪前后的yingbu脸谱。此外，计算所有脸谱的对比度取均值

In [2]:
# 计算图像的RMS Contrast
import cv2
import numpy as np

def compute_rms_contrast_per_channel(image_path):
    # 使用 OpenCV 读取图像（默认为 BGR 格式）
    img = cv2.imread(image_path)
    img = cv2.resize(img, (512, 512))
    if img is None:
        raise ValueError(f"Failed to load image: {image_path}")

    # 拆分通道（B, G, R）
    channels = cv2.split(img)

    # 存储每个通道的 RMS contrast
    rms_contrasts = {}

    for i, channel_name in enumerate(['B', 'G', 'R']):
        # 转为 float32 以避免溢出
        channel = channels[i].astype(np.float32)

        mean = np.mean(channel)
        rms = np.sqrt(np.mean((channel - mean) ** 2))
        rms_contrasts[channel_name] = rms

    return rms_contrasts

In [8]:
import os

face_dataset_path = "/data1/humw/Datasets/VGGFace2"

avg_rms_list = list()
rms_list = list()
for person_id in os.listdir(face_dataset_path):
    person_dir = os.path.join(face_dataset_path, person_id, "set_B")
    for img_name in os.listdir(person_dir):
        img_path = os.path.join(person_dir, img_name)
    
        contrast_dict = compute_rms_contrast_per_channel(img_path)

        # for ch, val in contrast_dict.items():
        #     print(f'RMS Contrast ({ch} channel): {val:.4f}')

        avg_rms = np.mean(list(contrast_dict.values()))
        # print(f'Average RMS Contrast: {avg_rms:.4f}')
        avg_rms_list.append(avg_rms)
        
        img = cv2.imread(img_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        mean = np.mean(gray)
        rms = np.sqrt(np.mean((gray - mean) ** 2))
        # print(f'RMS Contrast (Gray): {rms:.4f}')
        rms_list.append(rms)

print("Average RMS Contrast:", np.mean(avg_rms_list))
print("RMS Contrast (Gray):", np.mean(rms_list))

Average RMS Contrast: 64.99724
RMS Contrast (Gray): 63.83243546133735


In [9]:
print(len(avg_rms_list))
print(len(rms_list))

200
200


In [20]:
# 计算脸MIST的RMS Contrast
target_image_path = "/data1/humw/Codes/FaceOff/target_images/mist/MIST_0.png"

img = cv2.imread(target_image_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
mean = np.mean(gray)
rms = np.sqrt(np.mean((gray - mean) ** 2))
print(f'RMS Contrast (Gray): {rms:.4f}')

RMS Contrast (Gray): 110.0322


In [13]:
# 计算脸灰度图的RMS Contrast
target_image_path = "/data1/humw/Codes/FaceOff/target_images/gray/204_Gray_Uniformity_1.png"

img = cv2.imread(target_image_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
mean = np.mean(gray)
rms = np.sqrt(np.mean((gray - mean) ** 2))
print(f'RMS Contrast (Gray): {rms:.4f}')

RMS Contrast (Gray): 0.0000


In [5]:
# 计算yingbu的RMS Contrast
print("计算yingbu的RMS Contrast")
tgt_img_path = "/data1/humw/Codes/FaceOff/target_images/yingbu/yingbu0.png"
contrast_dict = compute_rms_contrast_per_channel(tgt_img_path)

for ch, val in contrast_dict.items():
    print(f'RMS Contrast ({ch} channel): {val:.4f}')
avg_rms = np.mean(list(contrast_dict.values()))
print(f'Average RMS Contrast: {avg_rms:.4f}')

# avg_rms_list.append(avg_rms)
img = cv2.imread(tgt_img_path)
img = cv2.resize(img, (512, 512))
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
mean = np.mean(gray)
rms = np.sqrt(np.mean((gray - mean) ** 2))
print(f'RMS Contrast (Gray): {rms:.4f}')

# 计算裁剪后的yingbu的RMS Contrast
print("计算裁剪后的yingbu的RMS Contrast")
tgt_img_path = "/data1/humw/Codes/My-Anti-DreamBooth/target_images/crop_yingbu.png"
contrast_dict = compute_rms_contrast_per_channel(tgt_img_path)

for ch, val in contrast_dict.items():
    print(f'RMS Contrast ({ch} channel): {val:.4f}')
avg_rms = np.mean(list(contrast_dict.values()))
print(f'Average RMS Contrast: {avg_rms:.4f}')

img = cv2.imread(tgt_img_path)
img = cv2.resize(img, (512, 512))
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
mean = np.mean(gray)
rms = np.sqrt(np.mean((gray - mean) ** 2))
print(f'RMS Contrast (Gray): {rms:.4f}')

# 计算六张脸谱的RMS Contrast
print("计算六张脸谱的RMS Contrast")
tgt_img_path_list = [
                    "/data1/humw/Codes/My-Anti-DreamBooth/target_images/caocao_qunyinghui_white.png",
                     "/data1/humw/Codes/My-Anti-DreamBooth/target_images/baozheng_chisangzhen_black.png",
                     "/data1/humw/Codes/My-Anti-DreamBooth/target_images/chengyaojin_jiajialou_green.png",
                     "/data1/humw/Codes/My-Anti-DreamBooth/target_images/guanyu_huarongdao_red.png",
                     "/data1/humw/Codes/My-Anti-DreamBooth/target_images/lumeng_zoumaicheng_blue.png",
                     "/data1/humw/Codes/FaceOff/target_images/yingbu/yingbu0.png"
                     ]
tgt_img_name_list = ["caocao", "baozheng", "chengyaojin", "guanyu", "lumeng", "yingbu"]

tgt_img_avg_rms_contrast_list = []
tgt_img_gray_rms_contrast_list = []
for ith, tgt_img_path in enumerate(tgt_img_path_list):
    contrast_dict = compute_rms_contrast_per_channel(tgt_img_path)
    print(f"tgt_img_name: {tgt_img_name_list[ith]}")
    # for ch, val in contrast_dict.items():
    #     print(f'RMS Contrast ({ch} channel): {val:.4f}')
    avg_rms = np.mean(list(contrast_dict.values()))
    print(f'Average RMS Contrast: {avg_rms:.4f}')
    tgt_img_avg_rms_contrast_list.append(avg_rms)
    
    img = cv2.imread(tgt_img_path)
    img = cv2.resize(img, (512, 512))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean = np.mean(gray)
    rms = np.sqrt(np.mean((gray - mean) ** 2))
    print(f'RMS Contrast (Gray): {rms:.4f}')
    tgt_img_gray_rms_contrast_list.append(rms)
print(f'Average RMS Contrast: {np.mean(tgt_img_avg_rms_contrast_list):.4f}')
print(f'Average RMS Contrast (Gray): {np.mean(tgt_img_gray_rms_contrast_list):.4f}')

计算yingbu的RMS Contrast
RMS Contrast (B channel): 88.5129
RMS Contrast (G channel): 79.1883
RMS Contrast (R channel): 72.4983
Average RMS Contrast: 80.0665
RMS Contrast (Gray): 75.1227
计算裁剪后的yingbu的RMS Contrast
RMS Contrast (B channel): 92.1284
RMS Contrast (G channel): 88.7663
RMS Contrast (R channel): 85.7962
Average RMS Contrast: 88.8970
RMS Contrast (Gray): 84.5837
计算六张脸谱的RMS Contrast
tgt_img_name: caocao
Average RMS Contrast: 92.2340
RMS Contrast (Gray): 91.3394
tgt_img_name: baozheng
Average RMS Contrast: 105.9300
RMS Contrast (Gray): 104.7434
tgt_img_name: chengyaojin
Average RMS Contrast: 102.0833
RMS Contrast (Gray): 92.2665
tgt_img_name: guanyu
Average RMS Contrast: 103.7775
RMS Contrast (Gray): 99.1333
tgt_img_name: lumeng
Average RMS Contrast: 97.8195
RMS Contrast (Gray): 96.0552
tgt_img_name: yingbu
Average RMS Contrast: 80.0665
RMS Contrast (Gray): 75.1227
Average RMS Contrast: 96.9851
Average RMS Contrast (Gray): 93.1101


- 目前的结果，如果用RMS Contrast，是可以从四种目标图像种筛选出MIST和MASK的，但是MIST的对比度是比MASK更高的，如何解释MASK在VGGFace2上的效果比MIST更好？
    - 因为MASK在人脸区域造成了更大的破坏，更符合评估指标 KO
- 还有一个问题，如何解释不同脸谱图像之间的效果差异，因为yingbu的对比度不是最高的，baozheng才是最高的，但是它在VGGFace2上的效果反而不如yingbu好？
    - 仍然用指标进行解释，指标差异其实不大，只看DreamBooth和LoRA两个模型

## CLIP的适应性
- 只能和图像相关，不能引入额外的条件，或者应该引入额外的一致条件
- 方式：计算MASK和MIST距离FACE的CILP编码的余弦相似度的平均值

In [1]:
# 加载数据集
from PIL import Image
import json
from pathlib import Path
from torchvision import transforms

dataset_dir = '/data1/humw/Datasets/VGGFace2'

def load_data(data_dir, image_size=512, resample=2):
    import numpy as np
    def image_to_numpy(image):
        return np.array(image).astype(np.uint8)
    # more robust loading to avoid loaing non-image files
    images = [] 
    for i in list(Path(data_dir).iterdir()):
        if not i.suffix in [".jpg", ".png", ".jpeg"]:
            continue
        else:
            images.append(image_to_numpy(Image.open(i).convert("RGB")))
    images = [Image.fromarray(i).resize((image_size, image_size), resample) for i in images]
    images = np.stack(images)
    # from B x H x W x C to B x C x H x W
    images = torch.from_numpy(images).permute(0, 3, 1, 2).float()
    assert images.shape[-1] == images.shape[-2]
    return images

train_aug = [
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(224),
    ]
tensorize_and_normalize = [
    transforms.Normalize([0.5*255]*3,[0.5*255]*3),
]
all_trans = train_aug + tensorize_and_normalize
all_trans = transforms.Compose(all_trans)
    
# 加载模型
from transformers.models.clip.modeling_clip import CLIPVisionModelWithProjection
import torch
import os
import torch.nn.functional as F

device = "cuda:0"
torch_dtype = torch.bfloat16
pretrained_model_name_or_path = '/data1/humw/Pretrains/clip-vit-large-patch14'
model = CLIPVisionModelWithProjection.from_pretrained(pretrained_model_name_or_path).to(device, dtype=torch_dtype)
model.to(torch_dtype)
id_embeds_dict = {}
# 获取图像编码
person_id_list = sorted(os.listdir(dataset_dir))
for person_id in person_id_list:
    print("person_id: {}".format(person_id))
    person_id_dir = os.path.join(dataset_dir, person_id, "set_B")
    clean_data = load_data(person_id_dir, 512, 2)
    original_data = clean_data.to(device).requires_grad_(False).to(dtype=torch_dtype)
    tran_original_data = all_trans(original_data)
    ori_embeds = model(tran_original_data, output_hidden_states=True).hidden_states[-2]
    id_embeds_dict[person_id] = ori_embeds
    
# 获取MIST图像编码
mist_data = load_data('/data1/humw/Codes/FaceOff/target_images/mist', 512, 2)
mist_data = mist_data.to(device).requires_grad_(False).to(dtype=torch_dtype)
tran_mist_data = all_trans(mist_data)
mist_embeds = model(tran_mist_data, output_hidden_states=True).hidden_states[-2]

# 获取yingbu图像编码
yingbu_data = load_data('/data1/humw/Codes/FaceOff/target_images/yingbu', 512, 2)
yingbu_data = yingbu_data.to(device).requires_grad_(False).to(dtype=torch_dtype)
tran_yingbu_data = all_trans(yingbu_data)
yingbu_embeds = model(tran_yingbu_data, output_hidden_states=True).hidden_states[-2]

mist_avg_cos = 0.0
yingbu_avg_cos = 0.0
# 计算两两之间的编码余弦损失距离，距离越大越好
for person_id_i in person_id_list:
    tmp_mist = -F.cosine_similarity(id_embeds_dict[person_id_i], mist_embeds, -1).mean() # 越近越小，最小-1，越远越大，最大1
    tmp_yingbu = -F.cosine_similarity(id_embeds_dict[person_id_i], yingbu_embeds, -1).mean()
    mist_avg_cos += tmp_mist
    yingbu_avg_cos += tmp_yingbu
mist_avg_cos = mist_avg_cos / len(person_id_list)
yingbu_avg_cos = yingbu_avg_cos / len(person_id_list)
print(mist_avg_cos, yingbu_avg_cos)

/data1/humw/anaconda3/envs/photomaker/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


person_id: n000050
person_id: n000057
person_id: n000058
person_id: n000061
person_id: n000063
person_id: n000068
person_id: n000076
person_id: n000080
person_id: n000087
person_id: n000088
person_id: n000089
person_id: n000090
person_id: n000097
person_id: n000098
person_id: n000103
person_id: n000104
person_id: n000105
person_id: n000110
person_id: n000138
person_id: n000139
person_id: n000142
person_id: n000145
person_id: n000146
person_id: n000150
person_id: n000154
person_id: n000161
person_id: n000164
person_id: n000170
person_id: n000171
person_id: n000172
person_id: n000176
person_id: n000179
person_id: n000180
person_id: n000181
person_id: n000184
person_id: n000185
person_id: n000187
person_id: n000188
person_id: n000190
person_id: n000215
person_id: n000217
person_id: n000220
person_id: n000221
person_id: n000223
person_id: n000225
person_id: n000228
person_id: n000234
person_id: n000236
person_id: n000238
person_id: n000243
tensor(-0.1787, device='cuda:0', dtype=torch.bfloa

In [3]:
print(mist_avg_cos.item())

print(yingbu_avg_cos.item())

-0.1787109375
-0.20703125


In [4]:
# 获取gray图像编码
gray_data = load_data('/data1/humw/Codes/FaceOff/target_images/gray', 512, 2)
gray_data = gray_data.to(device).requires_grad_(False).to(dtype=torch_dtype)
tran_gray_data = all_trans(gray_data)
gray_embeds = model(tran_gray_data, output_hidden_states=True).hidden_states[-2]

In [5]:
gray_avg_cos = 0.0
# 计算两两之间的编码余弦损失距离，距离越大越好
for person_id_i in person_id_list:
    tmp_gray = -F.cosine_similarity(id_embeds_dict[person_id_i], gray_embeds, -1).mean() # 越近越小，最小-1，越远越大，最大1
    tmp_gray = -F.cosine_similarity(id_embeds_dict[person_id_i], gray_embeds, -1).mean()
    gray_avg_cos += tmp_gray
    gray_avg_cos += tmp_gray
print(gray_avg_cos)

tensor(-25.6250, device='cuda:0', dtype=torch.bfloat16, grad_fn=<AddBackward0>)


- 越近越小，最小-1，越远越大，最大1